**FIN 585**  
**Diether**  
**Double Sort Portfolios**<br><br>

**1 Overview**

+ Goal $\rightarrow$ overview how to create double-sort portfolios

+ Double sort portfolios are very common in the academic literature, and also generally useful quant finance mmethod/tool.

+ Requires small extension of our standard coding tools $\rightarrow$ a three-way groupby instead of a two-way groupby.

+ Also cover some odds and ends about working with the CRSP data.

In [ ]:
import numpy as np
import pandas as pd
from finance_byu.summarize import summary

<br>

**2. Raw CRSP Data**

+ Datafile $\rightarrow$ raw CRSP data in the feather format (very good format $\rightarrow$ compact and speedy).

+ Raw CRSP data contains negative prices.

+ If no transaction at the end of the trading, CRSP reports average quotes from market makers.

+ If quote based price $\rightarrow$ reported as a negative price in CRSP.

+ Typically researchers don't care about this distinction.

+ Typically just take the absolute value of price to solve problem.

In [ ]:
df = pd.read_feather('12-mstk.ftr')
df.info()

In [ ]:
df[['prc','ret']].describe().round(3)

In [ ]:
df[['prc','ret']].quantile([0.05,0.1,0.15,0.20])

In [ ]:
df['prc'] = df['prc'].abs()
df['me']  = df.eval("prc*shr/1000.0").where(df.eval("prc*shr > 1e-6"))

df[['prc','ret','me']].quantile([0.05,0.1,0.15,0.20])

<br>

**3. Double Sort Portfolio Construction**

+ Sometimes you'll want to form portfolios based on two variables.

+ Example $\rightarrow$ forming based on lagged market-cap and momentum.<br><br>


**3.1 Breakpoints**

+ Need bins for both portfolio formation variables: momentum and market-cap.

+ Let's use NYSE breakpoints for market-cap.

+ We bin before splitting the sample so that the momentum breakpoints will be the same for both the small and large-cap stratification.

+ Called independent double sorting. $\leftarrow$ Fama French (1992)

+ Independent sorts make the comparisons across portfolio groupings more useful because the variation in momentum will be roughly the same across the portfolio groupings.

In [ ]:
df['prclag'] = df.groupby('permno')['prc'].shift(1)
df['melag'] = df.groupby('permno')['me'].shift(1)

df['logret'] = df.eval("log(1+ret)")
df['mom'] = df.groupby('permno')['logret'].rolling(11).sum().reset_index(drop=True)
df['mom'] = df.groupby('permno')['mom'].shift(2)

+ **NYSE Breakpoint Function**

  + First wrote function in annually rebalanced market-cap portfolio notebook (merging application).

  + Take a look at the notebook to review.

In [ ]:
def nyse_qcut(x,bp=[0.3,0.7]):
    bins = x.query("excd == 1")['melag'].quantile(bp).searchsorted(x['melag'])
    return pd.DataFrame(bins,index=x.index)

In [ ]:
df = df.query("mom == mom and melag == melag and 10 <= shrcd <= 11 and "
              "prclag >= 5").reset_index(drop=True)

df['bins'] = df.groupby('caldt')['mom'].transform(pd.qcut,5,labels=False)

In [ ]:
df['mebins'] = df.groupby('caldt',group_keys=False)[['excd','melag']].apply(nyse_qcut)
df

<br>

**3.2 Use Three Way Groupby to Double Sort**

+ Need to compute an equal-weight portfolio return for each date/market-cap/momentum bin combination.

+ Can accomplish that we a three-way groupby instead of the usual two-way.

+ Unstacking is a little bit trickier than usual.

In [ ]:
port = df.groupby(['caldt','mebins','bins'])['ret'].mean()*100
port

In [ ]:
port = port.unstack(level='bins')
port

In [ ]:
port.query("mebins == 0")

In [ ]:
port = df.groupby(['caldt','mebins','bins'])['ret'].mean()*100
port = port.unstack(level=['mebins','bins'])
port

In [ ]:
summary(port).loc[['mean','std','tstat']].round(3)